# Activity 1: RAGAS Evaluation with Cost Analysis

Compares the Fireworks-hosted RAG pipeline (`gpt-oss-20b` + `qwen3-embedding-8b`) against an
OpenAI-hosted equivalent (`gpt-4.1-mini` + `text-embedding-3-small`), using the **same**
retriever code path in `app/rag.py` (see the `provider` argument added to `_build_rag_graph`).

**Test set**: reused from `05_Synthetic_Data_Generation_for_RAG_Evals` &mdash; that lesson
generated and human-reviewed a RAGAS synthetic test set against the exact same
`cat-health-guide.pdf` (verified identical by MD5). Stored here as `data/eval_testset.json`.

**Metrics**: `context_precision` / `context_recall` (retrieval quality), `faithfulness`
(answer grounded in context), `answer_correctness` (end-to-end accuracy vs. ground truth).
The RAGAS judge model is fixed (OpenAI `gpt-4.1-mini`) across both runs so the comparison
isolates the pipeline under test, not the grader.

**Cost**: each run is traced to its own LangSmith project so token usage / cost can be
compared side by side in the LangSmith UI afterward.

In [1]:
import json
import os
import sys
import types
import warnings
import getpass

from dotenv import load_dotenv

load_dotenv()

# ragas 0.4.3 unconditionally imports langchain_community's VertexAI chat/llm classes,
# which langchain-community 0.4.x removed in its migration to standalone integration
# packages. We don't use VertexAI, so shim the missing symbols before importing ragas.
import langchain_community.llms as _cl_llms
if not hasattr(_cl_llms, "VertexAI"):
    _cl_llms.VertexAI = type("VertexAI", (), {})
_vertexai_shim = types.ModuleType("langchain_community.chat_models.vertexai")
_vertexai_shim.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = _vertexai_shim
warnings.filterwarnings("ignore", category=DeprecationWarning)

for key in ("FIREWORKS_API_KEY", "OPENAI_API_KEY", "LANGSMITH_API_KEY"):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(f"Enter your {key}: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")

/var/folders/n1/0xd6ct2s21n1fx6pj6xjx5940000gn/T/ipykernel_40271/2049520378.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.llms as _cl_llms


'true'

## Load the reused test set

In [2]:
with open("data/eval_testset.json") as f:
    test_set = json.load(f)

print(f"Loaded {len(test_set)} question/ground-truth pairs")
for row in test_set:
    print("-", row["question"][:90])

Loaded 4 question/ground-truth pairs
- What practical advice do the feline life stage guidelines give for getting more useful pat
- How do the recommendations for environmental enrichment for cats, together with the discus
- According to the 2021 AAHA/AAFP Feline Life Stage Guidelines, how should a veterinarian co
- What does the ACVIM guidance say about systemic hypertension in cats, and which other cat-


## Run both pipelines, one LangSmith project each

In [3]:
from langsmith import tracing_context
from app.rag import get_rag_graph

def run_pipeline(provider: str, langsmith_project: str, test_set: list[dict]) -> list[dict]:
    """Invoke the given provider's RAG graph over the test set.

    Returns rows shaped for ragas: user_input, response, retrieved_contexts, reference.

    Uses `tracing_context` (not `os.environ["LANGSMITH_PROJECT"] = ...`) to select the
    project. Reassigning the env var after the first trace has already been created does
    NOT redirect later traces in this process -- LangSmith resolves the project once and
    reuses it, so both providers silently landed in the same project the first time this
    was tried. `tracing_context` scopes the project correctly per call.
    """
    graph = get_rag_graph(provider)
    rows = []
    with tracing_context(project_name=langsmith_project):
        for item in test_set:
            result = graph.invoke({"question": item["question"]})
            rows.append({
                "user_input": item["question"],
                "response": result["response"],
                "retrieved_contexts": [doc.page_content for doc in result["context"]],
                "reference": item["ground_truth"],
            })
    return rows

In [4]:
fireworks_rows = run_pipeline("fireworks", "session10-rag-fireworks-v2", test_set)
print(f"Fireworks: {len(fireworks_rows)} rows collected")

Fireworks: 4 rows collected


In [5]:
openai_rows = run_pipeline("openai", "session10-rag-openai-v2", test_set)
print(f"OpenAI: {len(openai_rows)} rows collected")

OpenAI: 4 rows collected


## Score both with RAGAS\n\nJudge LLM/embeddings are fixed (OpenAI) across both evaluations.

In [6]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import context_precision, context_recall, faithfulness, answer_correctness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0))
judge_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))
metrics = [context_precision, context_recall, faithfulness, answer_correctness]

# Judge calls are their own cost center (the grading, not either app) -- keep them out of
# both providers' cost projects so provider cost numbers aren't inflated by grading calls.
with tracing_context(project_name="session10-rag-judge"):
    fireworks_scores = evaluate(
        dataset=EvaluationDataset.from_list(fireworks_rows),
        metrics=metrics,
        llm=judge_llm,
        embeddings=judge_embeddings,
    )
    openai_scores = evaluate(
        dataset=EvaluationDataset.from_list(openai_rows),
        metrics=metrics,
        llm=judge_llm,
        embeddings=judge_embeddings,
    )

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

## Side-by-side comparison

In [7]:
import pandas as pd

fw_means = fireworks_scores.to_pandas()[["context_precision", "context_recall", "faithfulness", "answer_correctness"]].mean()
oa_means = openai_scores.to_pandas()[["context_precision", "context_recall", "faithfulness", "answer_correctness"]].mean()

comparison = pd.DataFrame({
    "fireworks (gpt-oss-20b)": fw_means,
    "openai (gpt-4.1-mini)": oa_means,
})
comparison

,fireworks (gpt-oss-20b),openai (gpt-4.1-mini)
context_precision,0.979167,0.979167
context_recall,0.750000,0.937500
faithfulness,0.599129,1.000000
answer_correctness,0.450181,0.558722


## Next: cost, from LangSmith

RAGAS scores are above. For the cost half of Activity 1:

1. Open **smith.langchain.com** &rarr; projects `session10-rag-fireworks-v2`,
   `session10-rag-openai-v2`, and `session10-rag-judge` (the RAGAS grading calls are logged
   separately so they don't inflate either provider's cost).
2. OpenAI's cost shows up automatically (LangSmith has built-in pricing for it). For Fireworks,
   add custom per-token pricing under **Settings &rarr; Usage and Billing &rarr; Model pricing**
   for `accounts/fireworks/models/gpt-oss-20b` and `accounts/fireworks/models/qwen3-embedding-8b`
   (rates from Fireworks' pricing page), otherwise that dashboard only shows token counts.
3. Note avg. cost/query and avg. tokens/query for each project, then project to your expected
   scale (`cost_per_query * queries_per_day * 30`).
4. Drop the RAGAS table above and the cost numbers into your Loom, plus your take on the
   quality/cost trade-off.